# Python practice 10: errors, logging and testing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-10-errors-logging-and-testing.ipynb)

13 exercises on exceptions, context managers, logging, mocks, test runners and CLIs. Each exercise names the [reference entries](../learning/python-reference-10-errors-logging-and-testing.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 5 easy, 7 medium, 1 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Narrow exception handling](#exercise-1-narrow-exception-handling) | Easy | [`try`, `except`, `else`, `finally`](../learning/python-reference-10-errors-logging-and-testing.ipynb#try-except-else-finally) |
| 2 | [Validation errors with context](#exercise-2-validation-errors-with-context) | Medium | [Raising, chaining and custom exceptions](../learning/python-reference-10-errors-logging-and-testing.ipynb#raising-chaining-and-custom-exceptions) |
| 3 | [Chain the original error](#exercise-3-chain-the-original-error) | Easy | [Raising, chaining and custom exceptions](../learning/python-reference-10-errors-logging-and-testing.ipynb#raising-chaining-and-custom-exceptions) |
| 4 | [Add context notes](#exercise-4-add-context-notes) | Easy | [Raising, chaining and custom exceptions](../learning/python-reference-10-errors-logging-and-testing.ipynb#raising-chaining-and-custom-exceptions) |
| 5 | [Collect failures into an ExceptionGroup](#exercise-5-collect-failures-into-an-exceptiongroup) | Hard | [Exception groups and `except*`](../learning/python-reference-10-errors-logging-and-testing.ipynb#exception-groups-and-except) |
| 6 | [Retry with exponential backoff](#exercise-6-retry-with-exponential-backoff) | Medium | [Test doubles with `unittest.mock`](../learning/python-reference-10-errors-logging-and-testing.ipynb#test-doubles-with-unittestmock) |
| 7 | [Rollback context manager](#exercise-7-rollback-context-manager) | Medium | [`with`, `__enter__` and `__exit__`](../learning/python-reference-10-errors-logging-and-testing.ipynb#with-__enter__-and-__exit__) |
| 8 | [Temporary environment variables](#exercise-8-temporary-environment-variables) | Medium | [`contextlib`](../learning/python-reference-10-errors-logging-and-testing.ipynb#contextlib) |
| 9 | [Log rejects at the right level](#exercise-9-log-rejects-at-the-right-level) | Easy | [Loggers, levels and handlers](../learning/python-reference-10-errors-logging-and-testing.ipynb#loggers-levels-and-handlers) |
| 10 | [JSON log formatter](#exercise-10-json-log-formatter) | Medium | [Structured (JSON) logs](../learning/python-reference-10-errors-logging-and-testing.ipynb#structured-json-logs) |
| 11 | [Verify interactions with Mock](#exercise-11-verify-interactions-with-mock) | Easy | [Test doubles with `unittest.mock`](../learning/python-reference-10-errors-logging-and-testing.ipynb#test-doubles-with-unittestmock) |
| 12 | [A tiny parametrised test runner](#exercise-12-a-tiny-parametrised-test-runner) | Medium | [pytest essentials](../learning/python-reference-10-errors-logging-and-testing.ipynb#pytest-essentials) |
| 13 | [Command-line arguments with argparse](#exercise-13-command-line-arguments-with-argparse) | Medium | [Imports, packages and the `__main__` guard](../learning/python-reference-10-errors-logging-and-testing.ipynb#imports-packages-and-the-__main__-guard) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction


import logging
import os
from unittest.mock import Mock


def caught(fn):
    """Call fn and return the exception it raises, or None."""
    try:
        fn()
    except BaseException as error:  # noqa: BLE001
        return error
    return None


def run_with(manager, body):
    """Run body(value) inside `with manager as value`; return the exception raised, or None."""
    try:
        with manager as value:
            body(value)
    except Exception as error:  # noqa: BLE001
        return error
    return None


def capture_logs(fn, name="practice"):
    """Run fn(logger) and return its log lines as 'LEVEL:message'."""
    lines = []
    handler = logging.Handler()
    handler.emit = lambda record: lines.append(f"{record.levelname}:{record.getMessage()}")
    logger = logging.getLogger(name)
    logger.handlers, logger.propagate, logger.level = [handler], False, logging.DEBUG
    fn(logger)
    return lines

## Exercise 1: Narrow exception handling

**Difficulty:** Easy · about 5 min · **Practises:** [`try`, `except`, `else`, `finally`](../learning/python-reference-10-errors-logging-and-testing.ipynb#try-except-else-finally)

*Parsing user-supplied numbers without hiding real bugs.*

Write `to_number(value, default=None)` returning an int for integer strings, a float for decimal strings, the number unchanged for int/float input, and `default` for strings that do not parse or for None. Other types (lists, dicts) must still raise `TypeError`.

**Sample calls**

| call | expected |
| --- | --- |
| `to_number(" 42 ")` | `42` |
| `to_number("3.5")` | `3.5` |
| `to_number("abc", 0)` | `0` |

In [ ]:
def to_number(value, default=None):
    # your code here
    pass

In [ ]:
def to_number(value, default=None):
    if value is None:
        return default
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value
    if not isinstance(value, str):
        raise TypeError(f"unsupported type {type(value).__name__}")
    text = value.strip()
    try:
        return int(text)
    except ValueError:
        pass
    try:
        return float(text)
    except ValueError:
        return default

## Exercise 2: Validation errors with context

**Difficulty:** Medium · about 10 min · **Practises:** [Raising, chaining and custom exceptions](../learning/python-reference-10-errors-logging-and-testing.ipynb#raising-chaining-and-custom-exceptions)

*A small exception hierarchy lets callers catch pipeline failures broadly or specifically.*

Write `class PipelineError(Exception)`, `class ValidationError(PipelineError)` taking `(field, message)` with a `.field` attribute and `str()` of `field: message`, and `check_order(order)` raising `ValidationError` for a missing `id` ("required") or a non-positive `amount` ("must be positive"), else returning `order["id"]`.

**Sample calls**

| call | expected |
| --- | --- |
| `check_order({"id": 7, "amount": 10})` | `7` |
| `str(caught(lambda: check_order({"amount": 1})))` | `'id: required'` |
| `caught(lambda: check_order({"id": 1, "amount": 0})).field` | `'amount'` |

In [ ]:
class PipelineError(Exception):
    pass


class ValidationError:
    pass


def check_order(order):
    # your code here
    pass

In [ ]:
class PipelineError(Exception):
    pass

class ValidationError(PipelineError):
    def __init__(self, field, message):
        super().__init__(f"{field}: {message}")
        self.field = field

def check_order(order):
    if "id" not in order:
        raise ValidationError("id", "required")
    if order.get("amount", 0) <= 0:
        raise ValidationError("amount", "must be positive")
    return order["id"]

## Exercise 3: Chain the original error

**Difficulty:** Easy · about 5 min · **Practises:** [Raising, chaining and custom exceptions](../learning/python-reference-10-errors-logging-and-testing.ipynb#raising-chaining-and-custom-exceptions)

*Wrapping low-level errors keeps the API clean without losing the root cause.*

Write `class ConfigError(Exception)` and `load_config(text)` returning `json.loads(text)`, but raising `ConfigError("invalid config")` chained `from` the `JSONDecodeError`, and `ConfigError("config must be an object")` (no cause) when the JSON is not a dict.

**Sample calls**

| call | expected |
| --- | --- |
| `load_config('{"retries": 3}')` | `{'retries': 3}` |
| `str(caught(lambda: load_config("{")))` | `'invalid config'` |
| `type(caught(lambda: load_config("{")).__cause__).__name__` | `'JSONDecodeError'` |

In [ ]:
class ConfigError(Exception):
    pass


def load_config(text):
    # your code here
    pass

In [ ]:
import json

class ConfigError(Exception):
    pass

def load_config(text):
    try:
        value = json.loads(text)
    except json.JSONDecodeError as error:
        raise ConfigError("invalid config") from error
    if not isinstance(value, dict):
        raise ConfigError("config must be an object")
    return value

## Exercise 4: Add context notes

**Difficulty:** Easy · about 5 min · **Practises:** [Raising, chaining and custom exceptions](../learning/python-reference-10-errors-logging-and-testing.ipynb#raising-chaining-and-custom-exceptions)

*Errors deep in a batch are useless without the record that caused them.*

Write `with_context(fn, **context)` calling `fn()` and returning its result; if it raises, add a note `context: k1=v1, k2=v2` (keys sorted) to the exception and re-raise it unchanged.

**Sample calls**

| call | expected |
| --- | --- |
| `with_context(lambda: 5, row=1)` | `5` |
| `caught(lambda: with_context(lambda: 1 / 0, row=7, file="a.csv")).__notes__` | `['context: file=a.csv, row=7']` |
| `type(caught(lambda: with_context(lambda: {}["x"], k=1))).__name__` | `'KeyError'` |

In [ ]:
def with_context(fn, **context):
    # your code here
    pass

In [ ]:
def with_context(fn, **context):
    try:
        return fn()
    except Exception as error:
        error.add_note("context: " + ", ".join(f"{k}={v}" for k, v in sorted(context.items())))
        raise

## Exercise 5: Collect failures into an ExceptionGroup

**Difficulty:** Hard · about 15 min · **Practises:** [Exception groups and `except*`](../learning/python-reference-10-errors-logging-and-testing.ipynb#exception-groups-and-except)

*Validate everything, then report all failures at once, instead of stopping at the first.*

Write `run_all(tasks)` taking `(name, fn)` pairs; call each and return `{name: result}` if all succeed. If any raise, add a note `task <name>` to each error and raise `ExceptionGroup("<k> of <n> tasks failed", errors)`.

**Sample calls**

| call | expected |
| --- | --- |
| `run_all([("a", lambda: 1), ("b", lambda: 2)])` | `{'a': 1, 'b': 2}` |
| `str(caught(lambda: run_all([("a", lambda: 1 / 0), ("b", lambda: 2), ("c", lambda: int("x"))])))` | `'2 of 3 tasks failed (2 sub-exceptions)'` |
| `[type(e).__name__ for e in caught(lambda: run_all([("a", lambda: 1 / 0), ("c", lambda: int("x"))])).exceptions]` | `['ZeroDivisionError', 'ValueError']` |

In [ ]:
def run_all(tasks):
    # your code here
    pass

In [ ]:
def run_all(tasks):
    results, errors = {}, []
    for name, fn in tasks:
        try:
            results[name] = fn()
        except Exception as error:
            error.add_note(f"task {name}")
            errors.append(error)
    if errors:
        raise ExceptionGroup(f"{len(errors)} of {len(tasks)} tasks failed", errors)
    return results

## Exercise 6: Retry with exponential backoff

**Difficulty:** Medium · about 10 min · **Practises:** [Test doubles with `unittest.mock`](../learning/python-reference-10-errors-logging-and-testing.ipynb#test-doubles-with-unittestmock)

*Retry logic is easy to test when the sleep function is injected.*

Write `call_with_retry(fn, attempts=3, base=1.0, retry_on=(ConnectionError,), sleep=time.sleep)` calling `fn()` until it succeeds, sleeping `base * 2**(k-1)` seconds after the k-th failure; non-retryable errors propagate immediately; after the last attempt the error propagates. Checks pass a `Mock` as `sleep`.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda s: (call_with_retry(Mock(side_effect=[ConnectionError(), ConnectionError(), "ok"]), sleep=s), [c.args[0] for c in s.call_args_list]))(Mock())` | `('ok', [1.0, 2.0])` |
| `(lambda s: (call_with_retry(lambda: 1, sleep=s), s.call_count))(Mock())` | `(1, 0)` |
| `(lambda s: (raises(lambda: call_with_retry(Mock(side_effect=ConnectionError()), attempts=2, sleep=s), ConnectionError), s.call_count))(Mock())` | `(True, 1)` |

In [ ]:
def call_with_retry(fn, attempts=3, base=1.0, retry_on=(ConnectionError,), sleep=None):
    # your code here
    pass

In [ ]:
import time

def call_with_retry(fn, attempts=3, base=1.0, retry_on=(ConnectionError,), sleep=time.sleep):
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except retry_on:
            if attempt == attempts:
                raise
            sleep(base * 2 ** (attempt - 1))

## Exercise 7: Rollback context manager

**Difficulty:** Medium · about 10 min · **Practises:** [`with`, `__enter__` and `__exit__`](../learning/python-reference-10-errors-logging-and-testing.ipynb#with-__enter__-and-__exit__)

*All-or-nothing updates to in-memory state, the shape of a database transaction.*

Write class `Atomic(store)` usable as `with Atomic(d) as s:` that yields the dict itself and, if the block raises, restores the dict's contents to a deep copy taken on entry, then lets the exception propagate.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda d: (run_with(Atomic(d), lambda s: s.update(a=5)), d)[1])({"a": 1})` | `{'a': 5}` |
| `(lambda d: (type(run_with(Atomic(d), lambda s: (s.update(a=9, new=1), 1 / 0))).__name__, d))({"a": 1})` | `('ZeroDivisionError', {'a': 1})` |
| `(lambda d: (run_with(Atomic(d), lambda s: (s["x"].append(2), 1 / 0)), d)[1])({"x": [1]})` | `{'x': [1]}` |

In [ ]:
class Atomic:
    # your code here
    pass

In [ ]:
import copy

class Atomic:
    def __init__(self, store):
        self.store = store

    def __enter__(self):
        self.snapshot = copy.deepcopy(self.store)
        return self.store

    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None:
            self.store.clear()
            self.store.update(self.snapshot)
        return False

## Exercise 8: Temporary environment variables

**Difficulty:** Medium · about 10 min · **Practises:** [`contextlib`](../learning/python-reference-10-errors-logging-and-testing.ipynb#contextlib)

*Tests and scripts often need environment overrides that are always undone.*

Write a `@contextmanager` `temp_env(**values)` that sets the given environment variables (a value of None removes the variable) and restores the previous state on exit, even after an exception.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda seen: (run_with(temp_env(PX_STAGE="test"), lambda _: seen.append(os.environ.get("PX_STAGE"))), seen)[1])([])` | `['test']` |
| `(run_with(temp_env(PX_A="1"), lambda _: None), os.environ.get("PX_A"))` | `(None, None)` |
| `(lambda seen: (os.environ.__setitem__("PX_B", "x"), run_with(temp_env(PX_B=None), lambda _: seen.append(os.environ.get("PX_B"))), seen, os.environ.pop("PX_B", None))[2:])([])` | `([None], 'x')` |

In [ ]:
def temp_env(**values):
    # your code here
    pass

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def temp_env(**values):
    old = {k: os.environ.get(k) for k in values}
    try:
        for k, v in values.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v
        yield
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

## Exercise 9: Log rejects at the right level

**Difficulty:** Easy · about 5 min · **Practises:** [Loggers, levels and handlers](../learning/python-reference-10-errors-logging-and-testing.ipynb#loggers-levels-and-handlers)

*Pipelines should log a summary at INFO and each reject at WARNING, not print.*

Write `load(rows, logger)` returning the list of rows whose `amount` is a non-negative number. Log `rejected row %s: %s` at WARNING for each reject (row id and reason `missing amount` or `negative amount`) and `loaded %d of %d rows` at INFO at the end. Checks capture the log lines.

**Sample calls**

| call | expected |
| --- | --- |
| `capture_logs(lambda log: load([{"id": 1, "amount": 5}, {"id": 2}, {"id": 3, "amount": -1}], log))` | `['WARNING:rejected row 2: missing amount', 'WARNING:rejected row 3: negative amount', 'INFO:loaded 1 of 3 rows']` |
| `[r["id"] for r in load([{"id": 1, "amount": 0}, {"id": 2, "amount": -2}], Mock())]` | `[1]` |
| `capture_logs(lambda log: load([], log))` | `['INFO:loaded 0 of 0 rows']` |

In [ ]:
def load(rows, logger):
    # your code here
    pass

In [ ]:
def load(rows, logger):
    good = []
    for row in rows:
        amount = row.get("amount")
        if amount is None:
            logger.warning("rejected row %s: %s", row.get("id"), "missing amount")
        elif amount < 0:
            logger.warning("rejected row %s: %s", row.get("id"), "negative amount")
        else:
            good.append(row)
    logger.info("loaded %d of %d rows", len(good), len(rows))
    return good

## Exercise 10: JSON log formatter

**Difficulty:** Medium · about 10 min · **Practises:** [Structured (JSON) logs](../learning/python-reference-10-errors-logging-and-testing.ipynb#structured-json-logs)

*Structured logs make dashboards and alerts possible.*

Write class `JsonFormatter(logging.Formatter)` whose `format(record)` returns compact sorted-key JSON with `level`, `logger`, `message` (formatted with args) and any extra attributes that are not standard `LogRecord` attributes.

**Sample calls**

| call | expected |
| --- | --- |
| `JsonFormatter().format(logging.makeLogRecord({"name": "etl", "levelname": "INFO", "msg": "loaded %d rows", "args": (5,)}))` | `'{"level":"INFO","logger":"etl","message":"loaded 5 rows"}'` |
| `JsonFormatter().format(logging.makeLogRecord({"name": "etl", "levelname": "WARNING", "msg": "slow", "run_id": "r1", "seconds": 3.5}))` | `'{"level":"WARNING","logger":"etl","message":"slow","run_id":"r1","seconds":3.5}'` |
| `JsonFormatter().format(logging.makeLogRecord({"name": "x", "levelname": "ERROR", "msg": "boom"}))` | `'{"level":"ERROR","logger":"x","message":"boom"}'` |

In [ ]:
class JsonFormatter:
    # your code here
    pass

In [ ]:
import json
import logging

_STANDARD = set(vars(logging.makeLogRecord({}))) | {"message", "asctime"}

class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {"level": record.levelname, "logger": record.name, "message": record.getMessage()}
        payload.update({k: v for k, v in vars(record).items() if k not in _STANDARD})
        return json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str)

## Exercise 11: Verify interactions with Mock

**Difficulty:** Easy · about 5 min · **Practises:** [Test doubles with `unittest.mock`](../learning/python-reference-10-errors-logging-and-testing.ipynb#test-doubles-with-unittestmock)

*Alerting code is tested by checking what it would have sent.*

Write `alert_failures(results, send)` calling `send(channel, text)` once per failed result (`status != "ok"`) with channel `#data-alerts` and text `<job> failed: <error>`, and returning the number of alerts sent.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda m: (alert_failures([{"job": "a", "status": "ok"}, {"job": "b", "status": "failed", "error": "timeout"}], m), [c.args for c in m.call_args_list]))(Mock())` | `(1, [('#data-alerts', 'b failed: timeout')])` |
| `(lambda m: (alert_failures([{"job": "a", "status": "ok"}], m), m.called))(Mock())` | `(0, False)` |
| `alert_failures([], Mock())` | `0` |

In [ ]:
def alert_failures(results, send):
    # your code here
    pass

In [ ]:
def alert_failures(results, send):
    sent = 0
    for r in results:
        if r["status"] != "ok":
            send("#data-alerts", f"{r['job']} failed: {r.get('error')}")
            sent += 1
    return sent

## Exercise 12: A tiny parametrised test runner

**Difficulty:** Medium · about 10 min · **Practises:** [pytest essentials](../learning/python-reference-10-errors-logging-and-testing.ipynb#pytest-essentials)

*Understanding what `parametrize` and `pytest.raises` do by building them.*

Write `run_cases(func, cases)` where each case is `(case_id, args, expected)`; `expected` is either a value or an exception class meaning the call must raise it. Return the sorted list of failing case ids (wrong value, wrong or no exception, or an unexpected exception).

**Sample calls**

| call | expected |
| --- | --- |
| `run_cases(int, [("one", ("1",), 1), ("bad", ("x",), ValueError)])` | `[]` |
| `run_cases(abs, [("a", (-2,), 3), ("b", (2,), 2)])` | `['a']` |
| `run_cases(int, [("noerr", ("5",), ValueError)])` | `['noerr']` |

In [ ]:
def run_cases(func, cases):
    # your code here
    pass

In [ ]:
def run_cases(func, cases):
    failures = []
    for case_id, args, expected in cases:
        expects_error = isinstance(expected, type) and issubclass(expected, BaseException)
        try:
            result = func(*args)
        except Exception as error:
            if not (expects_error and isinstance(error, expected)):
                failures.append(case_id)
            continue
        if expects_error or result != expected:
            failures.append(case_id)
    return sorted(failures)

## Exercise 13: Command-line arguments with argparse

**Difficulty:** Medium · about 10 min · **Practises:** [Imports, packages and the `__main__` guard](../learning/python-reference-10-errors-logging-and-testing.ipynb#imports-packages-and-the-__main__-guard)

*Every scheduled job needs a predictable CLI.*

Write `parse_cli(argv)` using argparse with `--date` (required, ISO date), `--limit` (int, default 1000), `--dry-run` (flag) and `--tables` (one or more of `orders`, `refunds`; default both). Return a dict with keys `date` (ISO string), `limit`, `dry_run`, `tables`. Invalid input exits with `SystemExit`.

**Sample calls**

| call | expected |
| --- | --- |
| `parse_cli(["--date", "2026-11-27"])` | `{'date': '2026-11-27', 'limit': 1000, 'dry_run': False, 'tables': ['orders', 'refunds']}` |
| `parse_cli(["--date", "2026-01-02", "--limit", "5", "--dry-run", "--tables", "refunds"])` | `{'date': '2026-01-02', 'limit': 5, 'dry_run': True, 'tables': ['refunds']}` |
| `parse_cli(["--tables", "refunds", "orders", "--date", "2026-01-01"])["tables"]` | `['refunds', 'orders']` |

In [ ]:
def parse_cli(argv):
    # your code here
    pass

In [ ]:
import argparse
from datetime import date

def parse_cli(argv):
    parser = argparse.ArgumentParser(prog="job", exit_on_error=True)
    parser.add_argument("--date", type=date.fromisoformat, required=True)
    parser.add_argument("--limit", type=int, default=1000)
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--tables", nargs="+", choices=["orders", "refunds"], default=["orders", "refunds"])
    args = parser.parse_args(argv)
    return {"date": args.date.isoformat(), "limit": args.limit, "dry_run": args.dry_run, "tables": args.tables}

**Next:** look anything up in [Python reference 10: errors, logging and testing](../learning/python-reference-10-errors-logging-and-testing.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).